In [1]:
import os
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
import wrds
import pyarrow

/Users/nikolauswieland/Documents/masters_thesis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ASOF_DATE = "2024-12-31"     # universe as-of date
END_DATE  = "2024-12-31"     # last date to pull from CRSP
START_DATE = "1995-01-01"    # 30 years back from end 

FF3_PATH = "Data/FF3_Daily_Factors.csv"
OUT_FULL_PATH = "Data/crsp_ff3_daily_1996_2025.parquet"

# Chunk size for PERMNO batching (reduce if WRDS times out or something)
CHUNK_SIZE = 500


In [ ]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [ ]:
sql_permnos = f"""
select permno
from crsp.dsenames n
where n.shrcd in (10,11)
  and n.namedt <= '{ASOF_DATE}'
  and n.nameendt >= '{ASOF_DATE}'
  and permno in (
      select permno
      from crsp.dsf
      where date between '2019-01-01' and '2023-12-31'
      group by permno
      having count(ret) >= 1000
  )
"""



permnos = db.raw_sql(sql_permnos)
permno_list = permnos["permno"].astype(int).tolist()

print(f"PERMNO universe size: {len(permno_list):,}")


PERMNO universe size: 2,716


[10026, 10028, 10032, 10044, 10104, 10107, 10138, 10145, 10158, 10200]

In [5]:
chunks = [permno_list[i:i+CHUNK_SIZE] for i in range(0, len(permno_list), CHUNK_SIZE)]
dfs = []

for ch in tqdm(chunks, desc="Downloading CRSP dsf"):
    permno_str = ",".join(map(str, ch))
    sql_dsf = f"""
    select permno, date, ret, prc, vol, shrout
    from crsp.dsf
    where permno in ({permno_str})
      and date between '{START_DATE}' and '{END_DATE}'
    """
    tmp = db.raw_sql(sql_dsf, date_cols=["date"])
    dfs.append(tmp)

crsp_dsf = pd.concat(dfs, ignore_index=True)
crsp_dsf.sort_values(["permno", "date"], inplace=True)

print("CRSP dsf shape:", crsp_dsf.shape)

CRSP dsf shape: (13942604, 6)


In [ ]:
ff = pd.read_csv(FF3_PATH, skiprows=3)

# Rename first column to 'date'
ff.rename(columns={ff.columns[0]: "date"}, inplace=True)

# Keep only rows where date is numeric (droppping last row that states copyright details)
ff = ff[pd.to_numeric(ff["date"], errors="coerce").notnull()]

# Convert date properly
ff["date"] = pd.to_datetime(ff["date"].astype(int), format="%Y%m%d")

# Convert from % to decimal
for col in ["Mkt-RF", "SMB", "HML", "RF"]:
    ff[col] = ff[col].astype(float) / 100.0

# Standardize names
ff.rename(columns={
    "Mkt-RF": "mktrf",
    "SMB": "smb",
    "HML": "hml",
    "RF": "rf"
}, inplace=True)

ff = ff[["date","mktrf","smb","hml","rf"]]

print("FF cleaned shape:", ff.shape)
#ff.tail()


FF cleaned shape: (26151, 5)


,date,mktrf,smb,hml,rf
26146,2025-12-24,0.0029,0.0003,0.0001,0.0002
26147,2025-12-26,-0.0006,-0.0032,0.0009,0.0002
26148,2025-12-29,-0.0041,-0.0018,0.0007,0.0002
26149,2025-12-30,-0.0020,-0.0060,0.0028,0.0002
26150,2025-12-31,-0.0076,0.0007,-0.0009,0.0002


In [7]:
df = crsp_dsf.merge(ff, on="date", how="inner")

df.dropna(subset=["ret","rf","mktrf","smb","hml"], inplace=True)

df["excess_ret"] = df["ret"] - df["rf"]

df.sort_values(["permno","date"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Merged shape:", df.shape)
df.head()


Merged shape: (13748674, 11)


,permno,date,ret,prc,vol,shrout,mktrf,smb,hml,rf,excess_ret
0,10026,1995-01-03,-0.010753,11.5,18872.0,9408.0,-0.0026,-0.0097,0.0094,0.0002,-0.010953
1,10026,1995-01-04,0.0,11.5,33900.0,9408.0,0.0032,-0.0029,0.0047,0.0002,-0.0002
2,10026,1995-01-05,-0.01087,11.375,6202.0,9408.0,-0.0005,0.0004,0.0005,0.0002,-0.01107
3,10026,1995-01-06,0.010989,11.5,7510.0,9408.0,0.0018,-0.0001,0.0013,0.0002,0.010789
4,10026,1995-01-09,-0.01087,11.375,4000.0,9408.0,0.0008,0.0002,0.0013,0.0002,-0.01107


In [ ]:
df.to_parquet("Data/crsp_ff3_daily_1995_2024.parquet", index=False)